In [1]:
import spacy
from huggingface_hub import snapshot_download

from adept.components.registry import ComponentsRegistry
from adept.preprocess import Preprocess
from adept.postprocess import Postproccess
from adept.config import MODEL_DIR, logger

/Users/ben/Projects/ADEPT/Code/PrePub/ADEPT/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = snapshot_download(
    repo_id="Benscott/en_adept_ner_trf",
    local_dir=MODEL_DIR,
    revision='857dba16758cc25e50dfd7a9d42795f4747f6394'
)    
nlp = spacy.load(model)    

registry = ComponentsRegistry(nlp)       
registry.add_component('numeric', after="ner")
registry.add_component('anatomical_ner', after="ner")
registry.add_component('traits_ner', after="ner")
registry.add_component('traits_custom_ner', after="traits_ner")
registry.add_component('dimension_ner', after="numeric")
registry.add_component('measurement_rel', after="dimension_ner")  

Invalid model-index. Not loading eval results into CardData.
Fetching 24 files: 100%|███████████████████████████████████████████████████████████████████████████████████████████| 24/24 [00:00<00:00, 1173.48it/s]
/Users/ben/Projects/ADEPT/Code/PrePub/ADEPT/.venv/lib/python3.11/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_adept_ner_trf' (0.0.1) was trained with spaCy v3.4.4 and may not be 100% compatible with the current version (3.7.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
/Users/ben/Projects/ADEPT/Code/PrePub/ADEPT/.venv/lib/python3.11/site-packages/spacy_transformers/layers/hf_shim.py:137: UserWarning: Error loading saved torch state_dict with strict=True, likely due to differences between 'transformers' versions. Attempting to load with strict=False as a fallback...

If you

In [3]:
preprocess = Preprocess() 

In [91]:
text = """Perennials, 6–65+ cm (usually rhizomatous, sometimes stoloniferous). Stems 1(–4), erect, simple or branched, densely lanate-tomentose to glabrate. Leaves petiolate (proximally) or sessile (distally, weakly clasping and gradually reduced); blades oblong or lanceolate, 3.5–35+ cm × 5–35 mm, 1–2-pinnately lobed (ultimate lobes ± lanceolate, often arrayed in multiple planes), faces glabrate to sparsely tomentose or densely lanate. Heads 10–100+, in simple or compound, corymbiform arrays. Phyllaries 20–30 in ± 3 series, (light green, midribs dark green to yellowish, margins green to light or dark brown) ovate to lanceolate, abaxial faces tomentose. Receptacles convex; paleae lanceolate, 1.5–4 mm. Ray florets (3–)5–8, pistillate, fertile; corollas white or light pink to deep purple, laminae 1.5–3 × 1.5–3 mm. Disc florets 10–20; corollas white to grayish white, 2–4.5 mm. Cypselae 1–2 mm (margins broadly winged). 2n = 18, 27, 36, 45, 54, 63, 72 (including counts from Europe)."""

# text = """Stem thin, purple, not interlaced, making a few spirals around the host stem. Leaves fleshy, oblong, acute-obtuse, slightly purple to brown, inflexed, c. 1 x c. 0.4 mm. Flowers in clusters of 4-5, sessile or subsessile, 22-25 mm. Bracts leaf-like. Calyx lobes 4, 1.8-2 x 1-1.5 mm, glossy, fleshy, elliptical to spathulate, with a prominent reticulate pattern, mostly obtuse or acute-obtuse, tips glandular, tube 1-12 mm long. Petals 1.8-2 x 1-12 mm, 4, broadly elliptical, membranous, with reticulate pattern, acute-obtuse or obtuse; tube c. 1 mm long; scales very thin, membranous, hard to see, fringed, c. 0.3 mm long. Stamens 4, sessile, free filaments obsolete, fused filaments prominent in the petals; anther oblong-elliptical, basifixed, light-brown, to dark-brown, c. 0.3 x c. 0.4 mm. Ovary globular, not depressed, fleshy, brown, with thick base, 2.5-3 x c. 0.4 mm, style linear, narrowing towards the apex, brown, 2.5-3 mm long; stigma linear, dark brown, c. 0.2 mm long; ovules 4, orbicular, light brown to yellow, c. 0.2 mm in diameter. Fruit not seen."""

text = """Dioecious, perennial, 50-150 cm herb with a dense indumentum of stinging hairs. Stem angled. Leaves with 1.5-4 cm long petiole; lamina narrowly lanceolate to ovate, 5-12 cm long, 2.5-8 cm broad, cordate at the base, margin serrate, apex acute-acuminate; stipules free lateral, oblong-lanceolate, 8-12 mm long, ciliate. Racemes of cymes axillary, often longer than the subtending petiole, densely appressed pubescent, with scattered stinging hairs. Flowers pale-greenish, or whitish, bracteate; bracts of male flowers smaller than those of female flowers. Sepals pubescent. Achenes up to c. 1.5 mm long, ovoid-ellipsoid, pale green or greenish-brown."""



In [92]:
text = preprocess(text)  

In [93]:
doc = nlp(text)

/Users/ben/Projects/ADEPT/Code/PrePub/ADEPT/.venv/lib/python3.11/site-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):


In [99]:
from spacy.tokens import Span, Doc

from adept.traits import Traits
from adept.fields import Fields
from adept.utils.helpers import token_get_ent
from adept.config import logger



class Postproccess():
    
    """
    
    Run through the description Doc, parsing the different entities

    Returns:
        _type_: _description_
    """
    
    traits = Traits()

    PART_ALIASES = {
        "inflorescence": {
            "flower",
            "flowers",
        },
        "leaf": {
            "lamina",
            "foliage",
        },
        "petal": {
            "corolla",
            "corolla tube",
        },        
    }

    ALIAS_TO_CANONICAL = {
        canonical: canonical
        for canonical in PART_ALIASES
    }
    
    ALIAS_TO_CANONICAL.update({
        alias: canonical
        for canonical, aliases in PART_ALIASES.items()
        for alias in aliases
    })
    
    def __call__(self, doc, taxon_group):

        fields = Fields()
        for sent in doc.sents:
            part = self._sent_get_part(sent)       
            self._process_discrete_fields(sent, fields, part, taxon_group)
            self._process_custom_fields(sent, fields)
            if part:
                self._process_measurement_fields(sent, fields, part)
                self._process_colours_fields(sent, fields, part)
            self._process_numeric_fields(doc, sent, fields)
             
        logger.debug(fields.to_dict())   

        return fields
          
    @staticmethod
    def _sent_get_part(sent):
        return sent._.anatomical_part if sent._.anatomical_part else None
    
    def _process_colours_fields(self, sent: Doc, fields: Fields, part: str):
        colour_ents = self._filter_ents(sent, ['COLOUR'])
        for ent in colour_ents:
            fields.upsert(f'{part} colour', 'discrete', ent.lemma_)

    # def _get_rows(self)

    def search_part_alises(self, part: str) -> set[str]:

        canonical = self.ALIAS_TO_CANONICAL.get(part.strip().casefold())
    
        if canonical is None:
            return {part}
    
        return {canonical} | self.PART_ALIASES[canonical]    
            
    def _process_discrete_fields(self, sent: Doc, fields: Fields, part: str, taxon_group: str):        
        df = self.traits.get_discrete_traits(taxon_group)                
        trait_ents = set(self._filter_ents(sent, ['TRAIT', 'PART']))

        # trait_ents.add(part)
        # print(type(sent._.anatomical_part))

        
        # if part != 'stem':
        #     return

        print(sent)
        print(sent.ents)
        print(part)
        # for ent in sent.ents:
        #     print(ent)
        #     print(ent.label_)

        
        # Process entities     
        for ent in trait_ents:     

            # Special handling for bloody fig.XXX
            if ent.lemma_ == 'fig':
                try: 
                    next_token = ent.doc[ent.end]
                except IndexError:
                    pass
                else:
                    if next_token.text == '.': continue
                                      
            # Match on either term or character 
            mask = (((df.term == ent.lemma_) | (df.term == ent.text.lower()) | (df.character == ent.lemma_) | (df.character == ent.text.lower())))
            
            # BUGFIX - change logic here - previous: If the trait ent is a part, no need to filter on part        
            # Now - filter on part/part synonyms, plant and isna
            if part:
                parts = set(['plant'])
                parts.update(self.search_part_alises(part))
                mask &= ((df.part.isin(parts)) | (df.part.isna()))
            else:
                # No part - filter on na and plant
                mask &= ((df.part.isna()) | (df.part == 'plant'))
      
            rows = df[mask]    

            # if ent.text.lower() == 'leaf':
            #     print(rows)

            for row in rows.itertuples():   
                
                # if row.character == 'syconium':
                #     print('---')
                #     print(row.trait)
                #     print(row)
                #     print(ent)
                #     print(sent)
                                  
                fields.upsert(row.trait, 'discrete', row.character)   
                
    def _process_custom_fields(self, sent: Doc, fields: Fields):         
        for ent in sent.ents:
            if trait_value := ent._.get("trait_value"):
                fields.upsert(ent.label_.lower(), 'discrete', trait_value)        
    
    @staticmethod    
    def _get_sentence_measurements_filtered_by_part(sent, part: str):
        """
        Get sentence measurements, filtering for multiple parts in the sentence
        """
        measurements = [ent for ent in sent.ents if ent.label_ in ['DIMENSION', 'MEASUREMENT']]
        extra_sent_parts = [ent for ent in sent.ents if ent._.anatomical_part and ent._.anatomical_part != part]
        
        # If we have multiple parts and measurements in a sentence, filter the 
        # measurements to only those that proceed the first non-sentence part
        # e.g. Petals 2-3 cm. long and 1-2 mm wide with lamina 2 cm. => Petals 2-3 cm. long and 1-2 mm wide   
        if len(measurements) > 1 and extra_sent_parts:
            # Measurements whose ancestor includes extra sent parts
            # Petals 2-3 cm. long and 1-2 mm wide with 2 cm long lamina => 2cm
            measurements_with_ancestor_parts = [ent for ent in measurements if any(
                [token_get_ent(tok, 'PART') in extra_sent_parts for tok in ent.root.ancestors]
            )]                
            # We only want to use measurements which start before         
            max_start = min([p.start for p in extra_sent_parts + measurements_with_ancestor_parts])
            measurements = [ent for ent in measurements if ent.start < max_start]
            
        return measurements
        
    def _process_measurement_fields(self, sent: Doc, fields: Fields, part: str):
        for ent in self._get_sentence_measurements_filtered_by_part(sent, part):
            field_name = f'{part} measurement'         
            fields.upsert(field_name, 'measurement', ent)               

    @staticmethod
    def _parse_cardinal_part_subject(doc: Doc, cardinal_ent: Span): 
        """
        Use the dependency parse to locate the part/trait subject of a cardinal
        """         
        ent_labels = ['PART', 'TRAIT']
        # The root token of the cardinal ent - will contain the ancestral dependency
        token = cardinal_ent.root
        # If token is a numeric modifier, noun subject is the token head      
        if token.dep_ in 'nummod':
            if subject := token_get_ent(token.head, ent_labels):
                return subject

        # Try and locate a part/trait in the cardinal subtree     
        subtree = doc[token.left_edge.i: token.right_edge.i]
        part_ents = [ent for ent in subtree.ents if ent.label_ in ent_labels]
        if len(part_ents) == 1:
            return part_ents[0]
            
    def _process_numeric_fields(self, doc: Doc, sent: Doc, fields: Fields):        
        # Process numeric values (those not measurements or dimensions)     
        cardinal_ents = self._filter_ents(sent, ['CARDINAL'])       
        for cardinal_ent in cardinal_ents:
            if ent := self._parse_cardinal_part_subject(doc, cardinal_ent):  
                field_name = ent._.get("anatomical_part") or ent.lemma_
                fields.upsert(f'{field_name} number', 'numeric', cardinal_ent)
                
    @staticmethod
    def _filter_ents(sent: Span, label: list):
        return [ent for ent in sent.ents if ent.label_ in label]

postprocess = Postproccess()  

fields = postprocess(doc, 'angiosperm')
fields.to_dict()

Dioecious, perennial, 50-150 cm herb with a dense indumentum of stinging hairs.
[Dioecious, perennial, 50-150 cm, herb, indumentum, hairs]
plant
Stem angled.
[Stem]
stem
Leaves with 1.5-4 cm long petiole;
[Leaves, 1.5-4 cm, long, petiole]
leaf
lamina narrowly lanceolate to ovate, 5-12 cm long, 2.5-8 cm broad, cordate at the base, margin serrate, apex acute-acuminate;
[lamina, lanceolate, ovate, 5-12 cm, long, 2.5-8 cm, broad, cordate, serrate, acute, acuminate]
lamina
stipules free lateral, oblong-lanceolate, 8-12 mm long, ciliate.
[stipules, free, oblong-lanceolate, 8-12 mm, long, ciliate]
stipule
Racemes of cymes axillary, often longer than the subtending petiole, densely appressed pubescent, with scattered stinging hairs.
[Racemes, cymes, longer, petiole, appressed, pubescent, hairs]
raceme
Flowers pale-greenish, or whitish, bracteate;
[Flowers, greenish, whitish]
flower
bracts of male flowers smaller than those of female flowers.
[bracts, male, flowers, female, flowers]
bract
Sepal

OrderedDict([('habit', 'herb, erect leafy'),
             ('habitat', 'terrestrial'),
             ('life cycle', 'perennial'),
             ('dispersion axillary', 'hairs'),
             ('reproduction system', 'dioecious'),
             ('plant measurement.length.min', '50 cm'),
             ('plant measurement.length.max', '150 cm'),
             ('clonality', 'solitary plant'),
             ('perennial organ', 'stem'),
             ('leaf measurement.length.min', '1.5 cm'),
             ('leaf measurement.length.max', '4 cm'),
             ('leaf apex', 'acuminate, lanceolate, acute'),
             ('leaf shape', 'ovate, lanceolate, cordate'),
             ('leaf margin', 'serrate'),
             ('leaf base', 'cordate'),
             ('lamina measurement.length.min', '5 cm'),
             ('lamina measurement.length.max', '12 cm'),
             ('lamina measurement.width.min', '2.5 cm'),
             ('lamina measurement.width.max', '8 cm'),
             ('stipule measurement.leng